# 01c — Preprocess: Roboflow Person Detection Dataset

Downloads and preprocesses a person detection dataset from Roboflow.

Roboflow exports come pre-formatted in YOLOv8 format with `data.yaml`.

| | |
|---|---|
| **Source** | Roboflow |
| **Format** | YOLOv8 (TXT) |
| **Classes** | `person` |

In [ ]:
!pip install albumentations -q

import os, shutil, glob, random, yaml
import numpy as np
import matplotlib.pyplot as plt
import cv2
from collections import Counter

print('Setup done!')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/AI_TRAINING/GreenVision'
OUTPUT_DIR = '/content/dataset_roboflow'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f'Drive root: {DRIVE_ROOT}')

## 1. Download from Roboflow

In [ ]:
RAW_DIR = '/content/raw_roboflow'

if not os.path.exists(RAW_DIR):
    print('Downloading from Roboflow...')
    !curl -L "https://app.roboflow.com/ds/T6FSa305ce?key=yxgzptvuVX" > roboflow.zip
    !unzip -q roboflow.zip -d {RAW_DIR}
    !rm roboflow.zip
    print('Done!')
else:
    print(f'Already downloaded: {RAW_DIR}')

## 2. Inspect Structure

In [ ]:
# Walk directory tree
for root, dirs, files in os.walk(RAW_DIR):
    depth = root.replace(RAW_DIR, '').count(os.sep)
    indent = '  ' * depth
    folder = os.path.basename(root)
    print(f'{indent}{folder}/')
    if depth < 3:
        exts = Counter(os.path.splitext(f)[1].lower() for f in files)
        for ext, count in exts.most_common(10):
            print(f'{indent}  {ext or "(no ext)"}: {count} files')

# Check for data.yaml
yaml_files = glob.glob(os.path.join(RAW_DIR, '*.yaml')) + glob.glob(os.path.join(RAW_DIR, '*.yml'))
if yaml_files:
    print(f'\n--- data.yaml ---')
    print(open(yaml_files[0]).read())

# Sample a label file
label_files = glob.glob(os.path.join(RAW_DIR, '**', '*.txt'), recursive=True)
label_files = [f for f in label_files if not os.path.basename(f).lower().startswith(('readme', 'classes', 'data'))]
if label_files:
    print(f'\n--- Sample label ({os.path.basename(label_files[0])}) ---')
    print(open(label_files[0]).read()[:300])

## 3. Convert to Standard Format

Roboflow exports are usually already in YOLO format with train/valid/test splits. We normalize to a flat `images/` + `labels/` structure, remapping all classes to `0: person`.

In [ ]:
# Clean output dir
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(f'{OUTPUT_DIR}/images', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/labels', exist_ok=True)

# Find all images
all_images = glob.glob(os.path.join(RAW_DIR, '**', '*.jpg'), recursive=True) + \
             glob.glob(os.path.join(RAW_DIR, '**', '*.png'), recursive=True) + \
             glob.glob(os.path.join(RAW_DIR, '**', '*.jpeg'), recursive=True)

print(f'Total images found: {len(all_images)}')

copied = 0
skipped = 0

for img_path in all_images:
    base = os.path.splitext(os.path.basename(img_path))[0]
    ext = os.path.splitext(img_path)[1]
    img_dir = os.path.dirname(img_path)

    # Find matching label file
    # Roboflow usually mirrors: images/train/ -> labels/train/
    lbl_path = None
    for candidate in [
        img_path.replace('/images/', '/labels/').replace(ext, '.txt'),
        os.path.join(img_dir.replace('/images/', '/labels/'), base + '.txt'),
        os.path.join(img_dir, base + '.txt'),
    ]:
        if os.path.exists(candidate):
            lbl_path = candidate
            break

    if not lbl_path:
        skipped += 1
        continue

    # Read and remap labels to class 0 (person)
    with open(lbl_path) as f:
        lines = f.read().strip().split('\n')

    yolo_lines = []
    for line in lines:
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        # Remap any class ID to 0
        cx, cy, bw, bh = parts[1], parts[2], parts[3], parts[4]
        yolo_lines.append(f'0 {cx} {cy} {bw} {bh}')

    if not yolo_lines:
        skipped += 1
        continue

    # Handle duplicate names
    out_base = base
    out_img = f'{OUTPUT_DIR}/images/{out_base}{ext}'
    if os.path.exists(out_img):
        out_base = base + '_' + str(random.randint(1000, 9999))
        out_img = f'{OUTPUT_DIR}/images/{out_base}{ext}'
    out_lbl = f'{OUTPUT_DIR}/labels/{out_base}.txt'

    shutil.copy2(img_path, out_img)
    with open(out_lbl, 'w') as f:
        f.write('\n'.join(yolo_lines))
    copied += 1

print(f'Copied: {copied} image-label pairs')
print(f'Skipped: {skipped} (no label or empty)')

## 4. Augment (3x)

In [ ]:
import albumentations as A

img_dir = f'{OUTPUT_DIR}/images'
lbl_dir = f'{OUTPUT_DIR}/labels'

imgs = glob.glob(os.path.join(img_dir, '*.jpg')) + glob.glob(os.path.join(img_dir, '*.png'))
print(f'Original images: {len(imgs)}')

aug_flip = A.Compose([
    A.HorizontalFlip(p=1.0),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

aug_bc = A.Compose([
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.2, p=1.0),
    A.GaussNoise(var_limit=(10, 30), p=0.7),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

augmented = 0

for img_path in imgs:
    base = os.path.splitext(os.path.basename(img_path))[0]
    ext = os.path.splitext(img_path)[1]
    lbl_path = os.path.join(lbl_dir, base + '.txt')

    if not os.path.exists(lbl_path):
        continue

    image = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)

    with open(lbl_path) as f:
        lines = f.read().strip().split('\n')
    bboxes, labels = [], []
    for line in lines:
        parts = line.strip().split()
        if len(parts) == 5:
            labels.append(int(parts[0]))
            bboxes.append([float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])])

    if not bboxes:
        continue

    # Horizontal flip
    try:
        result = aug_flip(image=image, bboxes=bboxes, class_labels=labels)
        cv2.imwrite(os.path.join(img_dir, f'{base}_flip{ext}'),
                    cv2.cvtColor(result['image'], cv2.COLOR_RGB2BGR))
        with open(os.path.join(lbl_dir, f'{base}_flip.txt'), 'w') as f:
            for cls, bb in zip(result['class_labels'], result['bboxes']):
                f.write(f'{cls} {bb[0]:.6f} {bb[1]:.6f} {bb[2]:.6f} {bb[3]:.6f}\n')
        augmented += 1
    except Exception:
        pass

    # Brightness + contrast + noise
    try:
        result = aug_bc(image=image, bboxes=bboxes, class_labels=labels)
        cv2.imwrite(os.path.join(img_dir, f'{base}_bc{ext}'),
                    cv2.cvtColor(result['image'], cv2.COLOR_RGB2BGR))
        with open(os.path.join(lbl_dir, f'{base}_bc.txt'), 'w') as f:
            for cls, bb in zip(result['class_labels'], result['bboxes']):
                f.write(f'{cls} {bb[0]:.6f} {bb[1]:.6f} {bb[2]:.6f} {bb[3]:.6f}\n')
        augmented += 1
    except Exception:
        pass

final_imgs = glob.glob(os.path.join(img_dir, '*.jpg')) + glob.glob(os.path.join(img_dir, '*.png'))
print(f'\nAugmented: +{augmented} images')
print(f'Total: {len(final_imgs)} images ({len(final_imgs)/len(imgs):.1f}x)')

## 5. Stats & Preview

In [ ]:
img_dir = f'{OUTPUT_DIR}/images'
lbl_dir = f'{OUTPUT_DIR}/labels'

imgs = glob.glob(os.path.join(img_dir, '*.*'))
lbls = glob.glob(os.path.join(lbl_dir, '*.txt'))

total_boxes = 0
for lbl in lbls:
    with open(lbl) as f:
        total_boxes += len(f.readlines())

print('=' * 45)
print('  Roboflow Dataset Summary')
print('=' * 45)
print(f'  Images:         {len(imgs)}')
print(f'  Labels:         {len(lbls)}')
print(f'  Bounding boxes: {total_boxes}')
print(f'  Avg boxes/img:  {total_boxes/len(lbls):.1f}')
print('=' * 45)

# Sample visualization
CLASS_COLOR = (0, 255, 0)
originals = [p for p in imgs if '_flip' not in os.path.basename(p) and '_bc' not in os.path.basename(p)]
sample = random.sample(originals, min(12, len(originals)))

fig, axes = plt.subplots(3, 4, figsize=(20, 13))
for ax, img_path in zip(axes.flatten(), sample):
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    base = os.path.splitext(os.path.basename(img_path))[0]
    lbl = os.path.join(lbl_dir, base + '.txt')
    count = 0
    if os.path.exists(lbl):
        with open(lbl) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cx, cy, bw, bh = map(float, parts[1:5])
                    x1 = int((cx - bw/2) * w)
                    y1 = int((cy - bh/2) * h)
                    x2 = int((cx + bw/2) * w)
                    y2 = int((cy + bh/2) * h)
                    cv2.rectangle(img, (x1, y1), (x2, y2), CLASS_COLOR, 2)
                    count += 1
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f'{count} person(s)', fontsize=9)
plt.suptitle('Roboflow — Person Detection', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Save to Drive

In [ ]:
drive_output = os.path.join(DRIVE_ROOT, 'datasets', 'roboflow')
if os.path.exists(drive_output):
    shutil.rmtree(drive_output)
shutil.copytree(OUTPUT_DIR, drive_output)

print(f'Saved to: {drive_output}')
print(f'Images: {len(glob.glob(os.path.join(drive_output, "images", "*.*")))}')
print(f'Labels: {len(glob.glob(os.path.join(drive_output, "labels", "*.txt")))}')

---
## Done!

Preprocessed dataset saved to:
```
/content/drive/MyDrive/AI_TRAINING/GreenVision/datasets/roboflow/
  images/   — all images (original + augmented)
  labels/   — YOLO format labels (class 0 = person)
```

Ready to merge with other datasets or use for training.